In [3]:
import pandas as pd             
import numpy as np              
import seaborn as sns           
import matplotlib.pyplot as plt 
from scipy import stats         
import sqlite3                  
import os                       

# Настройка отображения: показывать все столбцы и 3 знака после запятой
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Часть 1: Анализ датасета mpg (Seaborn)")

# 1. Загрузка набора данных mpg из библиотеки Seaborn
df_mpg = sns.load_dataset('mpg')

# 2. Подсчет количества строк и столбцов
rows, cols = df_mpg.shape
print(f"\nКоличество строк: {rows}, Количество столбцов: {cols}")

# 3. Разведочный анализ данных (EDA)
# Разделяем числовые и категориальные переменные для удобства
numeric_cols = df_mpg.select_dtypes(include=['number']).columns
categorical_cols = df_mpg.select_dtypes(include=['object', 'category']).columns

print("\n Анализ числовых переменных ")
numeric_stats = []
for col in numeric_cols:
    stats_dict = {
        'Переменная': col,
        'Доля пропусков': f"{df_mpg[col].isna().mean():.2%}",
        'Минимум': df_mpg[col].min(),
        'Максимум': df_mpg[col].max(),
        'Среднее': df_mpg[col].mean(),
        'Медиана': df_mpg[col].median(),
        'Дисперсия': df_mpg[col].var(),
        'Квантиль 0.1': df_mpg[col].quantile(0.1),
        'Квантиль 0.9': df_mpg[col].quantile(0.9),
        'Квартиль 1 (Q1)': df_mpg[col].quantile(0.25),
        'Квартиль 3 (Q3)': df_mpg[col].quantile(0.75)
    }
    numeric_stats.append(stats_dict)
print(pd.DataFrame(numeric_stats).to_string(index=False))

print("\n Анализ категориальных переменных ")
categorical_stats = []
for col in categorical_cols:
    stats_dict = {
        'Переменная': col,
        'Доля пропусков': f"{df_mpg[col].isna().mean():.2%}",
        'Уникальных значений': df_mpg[col].nunique(),
        'Мода': df_mpg[col].mode()[0]  # Берем первое значение моды
    }
    categorical_stats.append(stats_dict)
print(pd.DataFrame(categorical_stats).to_string(index=False))

# 4. Формулировка и проверка статистических гипотез
print("\n Проверка статистических гипотез")

# Гипотеза 1: Средний расход топлива (mpg) автомобилей из США и Японии различается.
# Критерий: Двухвыборочный t-тест Стьюдента (независимые выборки).
# Обоснование: Сравниваем средние значения двух независимых групп числовых данных.
usa_mpg = df_mpg[df_mpg['origin'] == 'usa']['mpg'].dropna()
japan_mpg = df_mpg[df_mpg['origin'] == 'japan']['mpg'].dropna()
t_stat1, p_value1 = stats.ttest_ind(usa_mpg, japan_mpg, equal_var=False) # equal_var=False для теста Уэлча

print(f"\nГипотеза 1: t-статистика = {t_stat1:.4f}, p-значение = {p_value1:.4e}")
if p_value1 < 0.05:
    print("Вывод: Отвергаем нулевую гипотезу. Средний расход топлива автомобилей из США и Японии статистически значимо различается.")
else:
    print("Вывод: Не отвергаем нулевую гипотезу. Различий в среднем расходе топлива не выявлено.")

# Гипотеза 2: Между весом автомобиля (weight) и расходом топлива (mpg) есть линейная связь.
# Критерий: Коэффициент корреляции Пирсона.
# Обоснование: Обе переменные числовые, проверяем наличие линейной зависимости.
common_idx = df_mpg['weight'].dropna().index.intersection(df_mpg['mpg'].dropna().index)
pearson_corr, p_value2 = stats.pearsonr(df_mpg.loc[common_idx, 'weight'], df_mpg.loc[common_idx, 'mpg'])

print(f"\nГипотеза 2: Коэффициент корреляции Пирсона = {pearson_corr:.4f}, p-значение = {p_value2:.4e}")
if p_value2 < 0.05:
    print("Вывод: Отвергаем нулевую гипотезу. Существует статистически значимая отрицательная линейная связь между весом и расходом топлива.")
else:
    print("Вывод: Не отвергаем нулевую гипотезу. Линейная связь не выявлена.")

# 5. Кодирование категориальных переменных
# Используем OneHotEncoding для 'origin', так как это номинальный признак без порядка.
# drop_first=True убирает первый столбец, чтобы избежать мультиколлинеарности (лишней корреляции).
df_mpg_encoded = pd.get_dummies(df_mpg, columns=['origin'], drop_first=True)
print("\n Данные после OneHotEncoding (первые 5 строк) ")
df_mpg_encoded = pd.get_dummies(df_mpg, columns=['origin'], drop_first=True)
print("\n Данные после OneHotEncoding (первые 5 строк) ")
print(df_mpg_encoded[['mpg', 'origin_japan', 'origin_usa']].head())

# 6. Таблица корреляции признаков и целевого столбца
# Целевой столбец (target): 'mpg' (расход топлива), так как его чаще всего прогнозируют в таких задачах.
# Признаки (features): все остальные числовые и закодированные столбцы, кроме 'name'.
target = 'mpg'
features = [col for col in df_mpg_encoded.columns if col != target and col != 'name']

correlation_matrix = df_mpg_encoded[features + [target]].corr()
correlation_with_target = correlation_matrix[[target]].sort_values(by=target, ascending=False)

print("\n Таблица корреляции признаков с целевой переменной 'mpg' ")
print(correlation_with_target)
print("\nОбоснование: 'mpg' выбран как целевая переменная, так как эффективность расхода топлива является ключевым показателем для оптимизации. Остальные столбцы выступают признаками, влияющими на этот показатель.")


print("Часть 2 Вариант 4 - Indian Premier League SQLite Database")

# 7. Загрузка данных варианта самостоятельной работы
db_path = 'ipl_database.sqlite' # Путь к файлу базы данных (должен лежать в папке со скриптом)

if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    
    # Получаем список всех таблиц в базе данных
    tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
    tables = pd.read_sql_query(tables_query, conn)
    print(f"\nДоступные таблицы в базе данных: {tables['name'].tolist()}")
    
    # Для анализа выберем таблицу 'match' (или 'matches', проверим наличие)
    table_name = 'match' if 'match' in tables['name'].values else 'matches'
    
    if table_name in tables['name'].values:
        query = f"SELECT * FROM {table_name}"
        df_ipl = pd.read_sql_query(query, conn)
        print(f"\nЗагружена таблица '{table_name}'. Строк: {df_ipl.shape[0]}, Столбцов: {df_ipl.shape[1]}")
        
        # Разведочный анализ для числовых переменных IPL
        num_cols_ipl = df_ipl.select_dtypes(include=['number']).columns
        print("\n Статистика числовых переменных (IPL) ")
        for col in num_cols_ipl:
            print(f"\n{col}:")
            print(f"  Пропуски: {df_ipl[col].isna().mean():.2%}")
            print(f"  Мин: {df_ipl[col].min()}, Макс: {df_ipl[col].max()}")
            print(f"  Среднее: {df_ipl[col].mean():.2f}, Медиана: {df_ipl[col].median():.2f}")
            print(f"  Дисперсия: {df_ipl[col].var():.2f}")
            print(f"  Квантили (0.1, 0.9): {df_ipl[col].quantile(0.1):.2f}, {df_ipl[col].quantile(0.9):.2f}")
            print(f"  Квартили (Q1, Q3): {df_ipl[col].quantile(0.25):.2f}, {df_ipl[col].quantile(0.75):.2f}")
            
        # Разведочный анализ для категориальных переменных IPL
        cat_cols_ipl = df_ipl.select_dtypes(include=['object']).columns
        print("\n Статистика категориальных переменных (IPL) ")
        for col in cat_cols_ipl:
            print(f"\n{col}:")
            print(f"  Пропуски: {df_ipl[col].isna().mean():.2%}")
            print(f"  Уникальных значений: {df_ipl[col].nunique()}")
            print(f"  Мода: {df_ipl[col].mode()[0]}")
            
    else:
        print("Таблица 'match' или 'matches' не найдена в базе данных.")
    
    conn.close()
else:
    print(f"\nВнимание: Файл '{db_path}' не найден в текущей директории.")
    print("Пожалуйста, скачайте датасет 'Indian Premier League SQLite Database' с Kaggle,")
    print("распакуйте его и поместите файл .sqlite в папку с этим скриптом (main.py).")

Часть 1: Анализ датасета mpg (Seaborn)

Количество строк: 398, Количество столбцов: 9

 Анализ числовых переменных 
  Переменная Доля пропусков  Минимум  Максимум  Среднее  Медиана  Дисперсия  Квантиль 0.1  Квантиль 0.9  Квартиль 1 (Q1)  Квартиль 3 (Q3)
         mpg          0.00%    9.000    46.600   23.515   23.000     61.090        14.000        34.330           17.500           29.000
   cylinders          0.00%    3.000     8.000    5.455    4.000      2.893         4.000         8.000            4.000            8.000
displacement          0.00%   68.000   455.000  193.426  148.500  10872.199        90.000       350.000          104.250          262.000
  horsepower          1.51%   46.000   230.000  104.469   93.500   1481.569        67.000       157.700           75.000          126.000
      weight          0.00% 1613.000  5140.000 2970.425 2803.500 717140.991      1988.500      4275.200         2223.750         3608.000
acceleration          0.00%    8.000    24.800   15.568 